In [14]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("HotelBigDataSQL") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# Đọc file đã làm sạch từ HDFS của Hadoop
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("hdfs://localhost:9000/user/hotel/data/data_cleaned_hotel.csv")

# Tạo TempView
df.createOrReplaceTempView("hotel")
df.createOrReplaceTempView("hotel_bookings")

print("Đã nạp dữ liệu thành công! Sẵn sàng chạy 10 câu truy vấn SQL.")

Đã nạp dữ liệu thành công! Sẵn sàng chạy 10 câu truy vấn SQL.
Tổng số dòng : 119,390
Số cột       : 32


In [15]:
result_1 = spark.sql("""
    SELECT
        hotel,
        arrival_date_month,
        COUNT(*)                                          AS total_bookings,
        SUM(is_canceled)                                  AS total_canceled,
        ROUND(SUM(is_canceled) / COUNT(*) * 100, 2)       AS cancel_rate_pct
    FROM hotel
    GROUP BY
        hotel,
        arrival_date_month
    ORDER BY
        hotel,
        total_bookings DESC
""")

print("CÂU 1: PHÂN TÍCH MÙA VỤ ĐẶT PHÒNG")
result_1.show(30, truncate=False)

CÂU 1: PHÂN TÍCH MÙA VỤ ĐẶT PHÒNG
+------------+------------------+--------------+--------------+---------------+
|hotel       |arrival_date_month|total_bookings|total_canceled|cancel_rate_pct|
+------------+------------------+--------------+--------------+---------------+
|City Hotel  |August            |8983          |3602          |40.1           |
|City Hotel  |May               |8232          |3653          |44.38          |
|City Hotel  |July              |8088          |3306          |40.88          |
|City Hotel  |June              |7894          |3528          |44.69          |
|City Hotel  |October           |7605          |3268          |42.97          |
|City Hotel  |April             |7480          |3465          |46.32          |
|City Hotel  |September         |7400          |3110          |42.03          |
|City Hotel  |March             |6458          |2386          |36.95          |
|City Hotel  |February          |4965          |1901          |38.29          |
|City 

In [16]:
result_2 = spark.sql("""
    SELECT
        reserved_room_type,
        COUNT(*)                                                                        AS canceled_bookings,
        ROUND(SUM(adr * (stays_in_weekend_nights + stays_in_week_nights)), 2)           AS revenue_lost,
        DENSE_RANK() OVER (
            ORDER BY SUM(adr * (stays_in_weekend_nights + stays_in_week_nights)) DESC
        )                                                                               AS loss_rank
    FROM hotel
    WHERE is_canceled = 1
    GROUP BY reserved_room_type
    ORDER BY loss_rank
""")

print("\nCÂU 2: XẾP HẠNG THIỆT HẠI TÀI CHÍNH THEO LOẠI PHÒNG")
result_2.show(truncate=False)


CÂU 2: XẾP HẠNG THIỆT HẠI TÀI CHÍNH THEO LOẠI PHÒNG
+------------------+-----------------+-------------+---------+
|reserved_room_type|canceled_bookings|revenue_lost |loss_rank|
+------------------+-----------------+-------------+---------+
|A                 |33630            |1.010502932E7|1        |
|D                 |6102             |3277414.76   |2        |
|E                 |1914             |1270682.94   |3        |
|G                 |763              |741340.95    |4        |
|F                 |880              |670647.29    |5        |
|C                 |308              |287410.82    |6        |
|H                 |245              |241167.78    |7        |
|B                 |368              |133399.26    |8        |
|L                 |2                |144.0        |9        |
|P                 |12               |0.0          |10       |
+------------------+-----------------+-------------+---------+



In [17]:
result_3 = spark.sql("""
    SELECT
        CASE
            WHEN is_repeated_guest = 1 THEN 'Khach quen'
            WHEN is_repeated_guest = 0 THEN 'Khach moi'
        END                                                 AS guest_type,
        COUNT(*)                                            AS total_bookings,
        SUM(is_canceled)                                    AS total_canceled,
        ROUND(SUM(is_canceled) / COUNT(*) * 100, 2)         AS cancel_rate_pct
    FROM hotel
    GROUP BY
        CASE
            WHEN is_repeated_guest = 1 THEN 'Khach quen'
            WHEN is_repeated_guest = 0 THEN 'Khach moi'
        END
    ORDER BY total_bookings DESC
""")

print("\nCÂU 3: SO SÁNH HÀNH VI KHÁCH QUEN VÀ KHÁCH MỚI")
result_3.show(truncate=False)


CÂU 3: SO SÁNH HÀNH VI KHÁCH QUEN VÀ KHÁCH MỚI
+----------+--------------+--------------+---------------+
|guest_type|total_bookings|total_canceled|cancel_rate_pct|
+----------+--------------+--------------+---------------+
|Khach moi |115580        |43672         |37.79          |
|Khach quen|3810          |552           |14.49          |
+----------+--------------+--------------+---------------+



In [18]:
result_4 = spark.sql("""
    WITH lead_time_groups AS (
        SELECT
            *,
            CASE
                WHEN lead_time > 90 THEN 'Dat truoc >90 ngay'
                WHEN lead_time < 7  THEN 'Dat sat ngay (<7 ngay)'
                ELSE                     'Khac'
            END AS lead_time_group
        FROM hotel
    )
    SELECT
        lead_time_group,
        COUNT(*)                                                                    AS total_bookings,
        SUM(CASE WHEN is_canceled = 0 THEN 1 ELSE 0 END)                           AS successful_checkins,
        ROUND(SUM(CASE WHEN is_canceled = 0 THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS checkin_success_rate_pct
    FROM lead_time_groups
    GROUP BY lead_time_group
    ORDER BY total_bookings DESC
""")

print("\nCÂU 4: TÁC ĐỘNG CỦA THỜI GIAN ĐẶT PHÒNG TRƯỚC (LEAD TIME)")
result_4.show(truncate=False)


CÂU 4: TÁC ĐỘNG CỦA THỜI GIAN ĐẶT PHÒNG TRƯỚC (LEAD TIME)
+----------------------+--------------+-------------------+------------------------+
|lead_time_group       |total_bookings|successful_checkins|checkin_success_rate_pct|
+----------------------+--------------+-------------------+------------------------+
|Dat truoc >90 ngay    |51131         |25233              |49.35                   |
|Khac                  |49844         |33248              |66.7                    |
|Dat sat ngay (<7 ngay)|18415         |16685              |90.61                   |
+----------------------+--------------+-------------------+------------------------+



In [19]:
result_5 = spark.sql("""
    WITH channel_stats AS (
        SELECT
            market_segment,
            COUNT(*)                                        AS total_bookings,
            ROUND(AVG(adr), 2)                              AS avg_adr,
            SUM(is_canceled)                                AS total_canceled,
            ROUND(SUM(is_canceled) / COUNT(*) * 100, 2)     AS cancel_rate_pct
        FROM hotel
        GROUP BY market_segment
        HAVING COUNT(*) >= 100
    )
    SELECT *
    FROM channel_stats
    ORDER BY avg_adr DESC
""")

print("\nCÂU 5: RỦI RO TÀI CHÍNH THEO KÊNH PHÂN PHỐI (>= 100 booking)")
result_5.show(truncate=False)


CÂU 5: RỦI RO TÀI CHÍNH THEO KÊNH PHÂN PHỐI (>= 100 booking)
+--------------+--------------+-------+--------------+---------------+
|market_segment|total_bookings|avg_adr|total_canceled|cancel_rate_pct|
+--------------+--------------+-------+--------------+---------------+
|Online TA     |56477         |117.2  |20739         |36.72          |
|Direct        |12606         |115.45 |1934          |15.34          |
|Aviation      |237           |100.14 |52            |21.94          |
|Offline TA/TO |24219         |87.35  |8311          |34.32          |
|Groups        |19811         |79.48  |12097         |61.06          |
|Corporate     |5295          |69.36  |992           |18.73          |
|Complementary |743           |2.89   |97            |13.06          |
+--------------+--------------+-------+--------------+---------------+



In [20]:
result_6 = spark.sql("""
    SELECT
        reservation_status_date,
        ROUND(AVG(adr), 2)                                      AS daily_avg_adr,
        ROUND(
            AVG(AVG(adr)) OVER (
                ORDER BY reservation_status_date
                ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
            ), 2
        )                                                       AS rolling_7day_avg_adr
    FROM hotel
    GROUP BY reservation_status_date
    ORDER BY reservation_status_date ASC
""")

print("\nCÂU 6: GIÁ PHÒNG TRUNG BÌNH TRƯỢT 7 NGÀY")
result_6.show(30, truncate=False)


CÂU 6: GIÁ PHÒNG TRUNG BÌNH TRƯỢT 7 NGÀY
+-----------------------+-------------+--------------------+
|reservation_status_date|daily_avg_adr|rolling_7day_avg_adr|
+-----------------------+-------------+--------------------+
|2014-10-17             |62.8         |62.8                |
|2014-11-18             |0.0          |31.4                |
|2015-01-01             |62.06        |41.62               |
|2015-01-02             |9.63         |33.62               |
|2015-01-18             |0.0          |26.9                |
|2015-01-20             |76.5         |35.17               |
|2015-01-21             |37.3         |35.47               |
|2015-01-22             |116.57       |43.15               |
|2015-01-28             |244.0        |78.01               |
|2015-01-29             |66.0         |78.57               |
|2015-01-30             |56.81        |85.31               |
|2015-02-02             |79.74        |96.7                |
|2015-02-05             |155.83       |108.

In [21]:
query_7 = """
SELECT 
    total_of_special_requests,
    COUNT(*) AS total_bookings,
    SUM(is_canceled) AS total_canceled,
    ROUND(SUM(is_canceled) / COUNT(*) * 100, 2) AS cancel_rate_pct
FROM hotel
GROUP BY total_of_special_requests
ORDER BY total_of_special_requests ASC
"""

print("\nCÂU 7: TƯƠNG QUAN GIỮA YÊU CẦU ĐẶC BIỆT VÀ TỶ LỆ HỦY PHÒNG")
result_7 = spark.sql(query_7)
result_7.show(truncate=False)


CÂU 7: TƯƠNG QUAN GIỮA YÊU CẦU ĐẶC BIỆT VÀ TỶ LỆ HỦY PHÒNG
+-------------------------+--------------+--------------+---------------+
|total_of_special_requests|total_bookings|total_canceled|cancel_rate_pct|
+-------------------------+--------------+--------------+---------------+
|0                        |70318         |33556         |47.72          |
|1                        |33226         |7318          |22.02          |
|2                        |12969         |2866          |22.1           |
|3                        |2497          |446           |17.86          |
|4                        |340           |36            |10.59          |
|5                        |40            |2             |5.0            |
+-------------------------+--------------+--------------+---------------+



In [22]:
query_8 = """
WITH deposit_stats AS (
    SELECT 
        deposit_type,
        COUNT(*) AS total_bookings,
        SUM(is_canceled) AS total_canceled,
        ROUND(SUM(is_canceled) / COUNT(*) * 100, 2) AS cancel_rate_pct
    FROM hotel
    GROUP BY deposit_type
)
SELECT 
    deposit_type,
    total_bookings,
    total_canceled,
    cancel_rate_pct,
    CASE 
        WHEN cancel_rate_pct > 50 THEN 'Nguy co cao'
        WHEN cancel_rate_pct >= 20 AND cancel_rate_pct <= 50 THEN 'Trung binh'
        ELSE 'An toan'
    END AS risk_level
FROM deposit_stats
ORDER BY cancel_rate_pct DESC
"""

print("\nCÂU 8: TÁC ĐỘNG CỦA CHÍNH SÁCH ĐẶT CỌC ĐẾN HỦY PHÒNG")
result_8 = spark.sql(query_8)
result_8.show(truncate=False)


CÂU 8: TÁC ĐỘNG CỦA CHÍNH SÁCH ĐẶT CỌC ĐẾN HỦY PHÒNG
+------------+--------------+--------------+---------------+-----------+
|deposit_type|total_bookings|total_canceled|cancel_rate_pct|risk_level |
+------------+--------------+--------------+---------------+-----------+
|Non Refund  |14587         |14494         |99.36          |Nguy co cao|
|No Deposit  |104641        |29694         |28.38          |Trung binh |
|Refundable  |162           |36            |22.22          |Trung binh |
+------------+--------------+--------------+---------------+-----------+



In [23]:
query_9 = """
WITH waiting_stats AS (
    SELECT
        is_canceled,
        CASE
            WHEN is_canceled = 0 THEN 'Dat phong thanh cong'
            ELSE 'Da huy don'
        END AS booking_status,
        days_in_waiting_list
    FROM hotel_bookings
),
summary AS (
    SELECT
        booking_status,
        COUNT(*) AS total_bookings,
        SUM(CASE WHEN days_in_waiting_list > 0 THEN 1 ELSE 0 END) AS had_waiting,
        ROUND(AVG(days_in_waiting_list), 2) AS avg_waiting_days,
        ROUND(AVG(CASE WHEN days_in_waiting_list > 0
                       THEN days_in_waiting_list END), 2) AS avg_waiting_days_nonzero,
        MAX(days_in_waiting_list) AS max_waiting_days,
        ROUND(SUM(CASE WHEN days_in_waiting_list > 0 THEN 1 ELSE 0 END)
              * 100.0 / COUNT(*), 2) AS pct_had_to_wait
    FROM waiting_stats
    GROUP BY booking_status, is_canceled
)
SELECT
    booking_status,
    total_bookings,
    had_waiting,
    pct_had_to_wait,
    avg_waiting_days,
    avg_waiting_days_nonzero,
    max_waiting_days
FROM summary
ORDER BY booking_status DESC
"""

print("\nCÂU 9: PHÂN TÍCH TRẠNG THÁI CHỜ (WAITING LIST)")
result_9 = spark.sql(query_9)
result_9.show(truncate=False)


CÂU 9: PHÂN TÍCH TRẠNG THÁI CHỜ (WAITING LIST)
+--------------------+--------------+-----------+---------------+----------------+------------------------+----------------+
|booking_status      |total_bookings|had_waiting|pct_had_to_wait|avg_waiting_days|avg_waiting_days_nonzero|max_waiting_days|
+--------------------+--------------+-----------+---------------+----------------+------------------------+----------------+
|Dat phong thanh cong|75166         |1339       |1.78           |1.59            |89.25                   |379             |
|Da huy don          |44224         |2359       |5.33           |3.56            |66.82                   |391             |
+--------------------+--------------+-----------+---------------+----------------+------------------------+----------------+



In [24]:
query_10 = """
WITH guest_classified AS (
    SELECT
        is_canceled,
        adults,
        babies,
        stays_in_weekend_nights,
        stays_in_week_nights,
        hotel,
        adr,
        CASE
            WHEN (CAST(children AS INTEGER) > 0 OR babies > 0)
                THEN 'Gia dinh'
            WHEN (adults = 1 AND CAST(children AS INTEGER) = 0 AND babies = 0)
                THEN 'Don le'
            ELSE 'Nhom ban'
        END AS guest_group
    FROM hotel_bookings
    WHERE adults > 0
)
SELECT
    guest_group,
    hotel,
    COUNT(*) AS total_bookings,
    ROUND(AVG(stays_in_weekend_nights + stays_in_week_nights), 2) AS avg_total_nights,
    ROUND(AVG(CASE WHEN is_canceled = 0
                   THEN stays_in_weekend_nights + stays_in_week_nights END), 2)
        AS avg_nights_stayed,
    SUM(is_canceled) AS total_canceled,
    ROUND(SUM(is_canceled) * 100.0 / COUNT(*), 2) AS cancel_rate_pct,
    ROUND(AVG(adr), 2) AS avg_adr
FROM guest_classified
GROUP BY guest_group, hotel
ORDER BY guest_group, hotel
"""

print("\nCÂU 10: PHÂN TÍCH NHÓM KHÁCH HÀNG")
result_10 = spark.sql(query_10)
result_10.show(truncate=False)


CÂU 10: PHÂN TÍCH NHÓM KHÁCH HÀNG
+-----------+------------+--------------+----------------+-----------------+--------------+---------------+-------+
|guest_group|hotel       |total_bookings|avg_total_nights|avg_nights_stayed|total_canceled|cancel_rate_pct|avg_adr|
+-----------+------------+--------------+----------------+-----------------+--------------+---------------+-------+
|Don le     |City Hotel  |15564         |2.53            |2.31             |5372          |34.52          |93.52  |
|Don le     |Resort Hotel|7013          |3.0             |2.8              |1183          |16.87          |54.99  |
|Gia dinh   |City Hotel  |5180          |3.33            |3.09             |1778          |34.32          |152.5  |
|Gia dinh   |Resort Hotel|3929          |4.75            |4.41             |1397          |35.56          |161.14 |
|Nhom ban   |City Hotel  |58196         |3.06            |3.09             |25845         |44.41          |104.62 |
|Nhom ban   |Resort Hotel|29105      